# 13. Fusion RAG - 倒数排名融合

**复杂度:** ⭐⭐⭐

## 概述

**RAG-Fusion** 结合多个查询视角，使用倒数排名融合（Reciprocal Rank Fusion, RRF）来提高检索质量。它是对多查询 RAG（notebook 05）的增强版本。

### 问题

单一查询存在局限性：
- 可能因措辞而遗漏相关文档
- 无法捕捉复杂问题的多个方面
- 向量相似度对表述方式敏感

多查询 RAG 有所帮助，但简单的去重会丢失排名信息。

### 解决方案

RAG-Fusion 在多查询 RAG 的基础上进行了改进：
1. 生成多个查询变体（类似多查询）
2. 为每个查询检索文档
3. **使用倒数排名融合**智能地合并结果
4. 基于综合得分重新排名文档

### 倒数排名融合（RRF）

RRF 是一种强大的排名聚合方法：

```
RRF_score(doc) = Σ (1 / (k + rank_i))
```

其中：
- `k` = 常数（通常为 60）
- `rank_i` = 文档在第 i 个查询结果中的排名
- 对文档出现的所有查询进行求和

**示例：**
```
查询 1 排名: [A, B, C, D]
查询 2 排名: [C, A, E, F]
查询 3 排名: [B, C, A, G]

文档 A:
  - 在 Q1 中排名第 1: 1/(60+1) = 0.0164
  - 在 Q2 中排名第 2: 1/(60+2) = 0.0161
  - 在 Q3 中排名第 3: 1/(60+3) = 0.0159
  - 总 RRF 得分: 0.0484

文档 C:
  - 在 Q1 中排名第 3: 1/(60+3) = 0.0159
  - 在 Q2 中排名第 1: 1/(60+1) = 0.0164
  - 在 Q3 中排名第 2: 1/(60+2) = 0.0161
  - 总 RRF 得分: 0.0484
```

**为什么 RRF 有效：**
- ✅ 有利于出现在多个查询结果中的文档
- ✅ 同时考虑频率和排名位置
- ✅ 无需分数归一化
- ✅ 对异常值具有鲁棒性

### 流程

```
查询 → 生成 N 个替代查询 → 为每个查询检索
    → 应用 RRF 算法 → 重新排名文档 → 生成答案
```

### Fusion RAG vs 多查询 RAG

| 方面 | 多查询 RAG | Fusion RAG |
|--------|-----------------|------------|
| 查询生成 | ✅ 多个查询 | ✅ 多个查询 |
| 检索 | ✅ 并行 | ✅ 并行 |
| 融合方法 | 简单去重 | **RRF 算法** |
| 排名 | 丢失 | **保留并合并** |
| 质量 | 良好 | **更好** |
| 复杂度 | 低 | 中等 |

### 何时使用

✅ **适用于：**
- 复杂的、多层面的问题
- 具有多种有效解释的查询
- 当排名质量很重要时
- 当你需要稳健的检索时

❌ **不适用于：**
- 简单的事实查找
- 延迟关键的应用
- API 预算有限

### 权衡

**优点：**
- ✅ 比单查询 RAG 更好的检索效果
- ✅ 比多查询 RAG 更 sophisticated
- ✅ 稳健的排名算法
- ✅ 捕捉多样化的视角

**缺点：**
- ❌ 更高的延迟（多次检索）
- ❌ 更多的 API 调用（查询生成）
- ❌ 更复杂的实现

---

## 实现

让我们逐步构建 RAG-Fusion。

## 1. 设置和导入

In [1]:
import sys
import time
from pathlib import Path
from collections import defaultdict
from typing import List

# Add parent directory to path for imports
sys.path.append(str(Path("../..").resolve()))

from shared.config import create_chat_model, create_embeddings, DEFAULT_MODEL, DEFAULT_TEMPERATURE, DEFAULT_VISION_MODEL
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

from shared.config import (
    verify_api_key,
    DEFAULT_MODEL,
    DEFAULT_TEMPERATURE,
    HF_EMBEDDING_MODEL,
    VECTOR_STORE_DIR,
)
from shared.loaders import load_and_split
from shared.prompts import (
    FUSION_QUERY_GENERATION_PROMPT,
    FUSION_RAG_ANSWER_PROMPT,
)
from shared.utils import (
    format_docs,
    print_section_header,
    load_vector_store,
    save_vector_store,
)

# Verify API key
verify_api_key()

print("✓ All imports successful")
print(f"✓ Using model: {DEFAULT_MODEL}")
print(f"✓ Using embeddings: {HF_EMBEDDING_MODEL}")

OK deepseek API key: LOADED
  Preview: sk-0501...20ae
  Base URL: https://api.deepseek.com
✓ All imports successful
✓ Using model: deepseek-v4-flash
✓ Using embeddings: sentence-transformers/all-MiniLM-L6-v2


## 2. 加载文档并创建向量存储

In [2]:
from langchain_community.vectorstores import FAISS

print_section_header("Loading Documents and Vector Store")

# Load and split documents (returns tuple: original_docs, chunks)
_, docs = load_and_split(
    chunk_size=1000,
    chunk_overlap=200,
)

print(f"\n✓ Loaded {len(docs)} chunks")

# Initialize embeddings
embeddings = create_embeddings()

# Load or create vector store
store_path = VECTOR_STORE_DIR / "fusion_rag"
vectorstore = load_vector_store(store_path, embeddings)

if vectorstore is None:
    print("\nCreating vector store...")
    vectorstore = FAISS.from_documents(docs, embeddings)
    save_vector_store(vectorstore, store_path)
    print("✓ Vector store created and saved")
else:
    print("✓ Loaded existing vector store")

# Create retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
print("✓ Retriever ready")


LOADING DOCUMENTS AND VECTOR STORE

Loading 4 documents from web...
  - https://python.langchain.com/docs/use_cases/question_answering/
  - https://python.langchain.com/docs/modules/data_connection/retrievers/
  - https://python.langchain.com/docs/modules/model_io/llms/
  - https://python.langchain.com/docs/use_cases/chatbots/
✓ Loaded 4 documents
✓ Added custom metadata to all documents
Splitting documents...
  - Chunk size: 1000
  - Chunk overlap: 200
✓ Created 165 chunks

  Sample chunk:
    - Length: 995 chars
    - Source: https://python.langchain.com/docs/use_cases/question_answering/
    - Preview: Build a RAG agent with LangChain - Docs by LangChainSkip to main contentJoin us May 13th & May 14th at Interrupt, the Agent Conference by LangChain. B...

✓ Loaded 165 chunks


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loaded vector store from D:\ZKS_Data\AI\langchain-rag-tutorial\data\vector_stores\fusion_rag
✓ Loaded existing vector store
✓ Retriever ready


## 3. 实现倒数排名融合

RAG-Fusion 的核心算法。

In [3]:
def reciprocal_rank_fusion(
    results_list: List[List[Document]],
    k: int = 60,
) -> List[tuple[Document, float]]:
    """
    Apply Reciprocal Rank Fusion to combine multiple ranked lists.
    
    Args:
        results_list: List of ranked document lists (one per query)
        k: RRF constant (default 60, as per literature)
    
    Returns:
        List of (document, score) tuples, sorted by score descending
    """
    # Track scores for each unique document
    doc_scores = defaultdict(float)
    doc_objects = {}  # Map content to document object
    
    # Process each query's results
    for results in results_list:
        for rank, doc in enumerate(results, start=1):
            # Use page_content as unique identifier
            doc_key = doc.page_content
            
            # Calculate RRF score contribution
            score = 1.0 / (k + rank)
            doc_scores[doc_key] += score
            
            # Store document object (in case of duplicates, keep first)
            if doc_key not in doc_objects:
                doc_objects[doc_key] = doc
    
    # Sort by score descending
    ranked_docs = [
        (doc_objects[doc_key], score)
        for doc_key, score in sorted(
            doc_scores.items(),
            key=lambda x: x[1],
            reverse=True,
        )
    ]
    
    return ranked_docs


print("✓ Reciprocal Rank Fusion function defined")

# Test the function
print("\nExample RRF calculation:")
print("-" * 80)
print("Query 1 ranks: [A(rank=1), B(rank=2), C(rank=3)]")
print("Query 2 ranks: [C(rank=1), A(rank=2), D(rank=3)]")
print("Query 3 ranks: [B(rank=1), C(rank=2), A(rank=3)]")
print("\nRRF scores (k=60):")
print("  Doc A: 1/61 + 1/62 + 1/63 = 0.0164 + 0.0161 + 0.0159 = 0.0484")
print("  Doc B: 1/62 + 1/61 = 0.0161 + 0.0164 = 0.0325")
print("  Doc C: 1/63 + 1/61 + 1/62 = 0.0159 + 0.0164 + 0.0161 = 0.0484")
print("  Doc D: 1/63 = 0.0159")
print("\nFinal ranking: A=C (tie) > B > D")

✓ Reciprocal Rank Fusion function defined

Example RRF calculation:
--------------------------------------------------------------------------------
Query 1 ranks: [A(rank=1), B(rank=2), C(rank=3)]
Query 2 ranks: [C(rank=1), A(rank=2), D(rank=3)]
Query 3 ranks: [B(rank=1), C(rank=2), A(rank=3)]

RRF scores (k=60):
  Doc A: 1/61 + 1/62 + 1/63 = 0.0164 + 0.0161 + 0.0159 = 0.0484
  Doc B: 1/62 + 1/61 = 0.0161 + 0.0164 = 0.0325
  Doc C: 1/63 + 1/61 + 1/62 = 0.0159 + 0.0164 + 0.0161 = 0.0484
  Doc D: 1/63 = 0.0159

Final ranking: A=C (tie) > B > D


## 4. 构建查询生成链

In [4]:
print_section_header("Building Query Generation Chain")

# Initialize LLM
llm = create_chat_model(
    model=DEFAULT_MODEL,
    temperature=DEFAULT_TEMPERATURE,
)

# Create query generation chain
query_gen_chain = FUSION_QUERY_GENERATION_PROMPT | llm | StrOutputParser()

print("✓ Query generation chain created")

# Test query generation
test_query = "What is LCEL in LangChain?"
print(f"\nTest query: {test_query}")
print("\nGenerated alternatives:")
print("-" * 80)

alternative_queries_text = query_gen_chain.invoke({
    "question": test_query,
    "num_queries": 3,
})
print(alternative_queries_text)


BUILDING QUERY GENERATION CHAIN



langchain-openai injected a custom httpx transport to apply `http_socket_options`, which disables httpx's proxy auto-detection (system proxy configuration detected). Set `LANGCHAIN_OPENAI_TCP_KEEPALIVE=0` or pass `http_socket_options=()` to restore default proxy behavior, or supply `openai_proxy` / your own `http_client` / `http_async_client` to take full control.


✓ Query generation chain created

Test query: What is LCEL in LangChain?

Generated alternatives:
--------------------------------------------------------------------------------
1. Can you explain the meaning of LCEL in the LangChain framework?
2. What does LCEL stand for, and how is it used to build chains in LangChain?
3. In LangChain, what is the LangChain Expression Language (LCEL) and how does it simplify pipeline construction?


## 5. 实现 Fusion RAG 检索器

In [5]:
def fusion_retriever(
    query: str,
    retriever,
    llm,
    num_queries: int = 4,
    k_final: int = 6,
    rrf_k: int = 60,
    verbose: bool = False,
) -> List[Document]:
    """
    Fusion RAG retriever with RRF.
    
    Args:
        query: Original user query
        retriever: LangChain retriever
        llm: LLM for query generation
        num_queries: Number of alternative queries to generate
        k_final: Number of final documents to return
        rrf_k: RRF constant
        verbose: Print debug information
    
    Returns:
        List of re-ranked documents
    """
    if verbose:
        print(f"\n[Fusion Retriever] Original query: {query}")
    
    # 1. Generate alternative queries
    query_gen_chain = FUSION_QUERY_GENERATION_PROMPT | llm | StrOutputParser()
    alternatives_text = query_gen_chain.invoke({
        "question": query,
        "num_queries": num_queries - 1,  # -1 because we include original
    })
    
    # Parse alternative queries
    alternative_queries = [
        line.strip()
        for line in alternatives_text.split("\n")
        if line.strip() and any(char.isalpha() for char in line)
    ]
    
    # Remove numbering if present (e.g., "1. Query" -> "Query")
    alternative_queries = [
        q.split(".", 1)[1].strip() if q[0].isdigit() and "." in q else q
        for q in alternative_queries
    ]
    
    # Include original query
    all_queries = [query] + alternative_queries[:num_queries-1]
    
    if verbose:
        print(f"\n[Fusion Retriever] Generated {len(all_queries)} queries:")
        for i, q in enumerate(all_queries, 1):
            print(f"  {i}. {q}")
    
    # 2. Retrieve documents for each query
    all_results = []
    for q in all_queries:
        results = retriever.invoke(q)
        all_results.append(results)
        if verbose:
            print(f"\n[Fusion Retriever] Query '{q[:50]}...' retrieved {len(results)} docs")
    
    # 3. Apply Reciprocal Rank Fusion
    fused_results = reciprocal_rank_fusion(all_results, k=rrf_k)
    
    if verbose:
        print("\n[Fusion Retriever] After RRF, top 3 scores:")
        for i, (doc, score) in enumerate(fused_results[:3], 1):
            print(f"  {i}. Score: {score:.4f} | Preview: {doc.page_content[:60]}...")
    
    # 4. Return top-k documents
    final_docs = [doc for doc, score in fused_results[:k_final]]
    
    if verbose:
        print(f"\n[Fusion Retriever] Returning top {len(final_docs)} documents")
    
    return final_docs


print("✓ Fusion retriever function defined")

✓ Fusion retriever function defined


## 6. 构建 Fusion RAG 链

In [6]:
# Create a custom runnable for fusion retrieval
from langchain_core.runnables import RunnableLambda

print_section_header("Building Fusion RAG Chain")

fusion_retriever_runnable = RunnableLambda(
    lambda x: fusion_retriever(
        query=x if isinstance(x, str) else str(x),
        retriever=retriever,
        llm=llm,
        num_queries=4,
        k_final=6,
        verbose=False,
    )
)

# Build the RAG chain
fusion_rag_chain = (
    {
        "context": fusion_retriever_runnable | format_docs,
        "input": RunnablePassthrough(),
        "original_query": RunnablePassthrough(),
        "num_queries": lambda _: 4,
        "num_docs": lambda _: 6,
    }
    | FUSION_RAG_ANSWER_PROMPT
    | llm
    | StrOutputParser()
)

print("✓ Fusion RAG chain created")


BUILDING FUSION RAG CHAIN

✓ Fusion RAG chain created


## 7. 测试 Fusion RAG

In [7]:
print_section_header("Testing Fusion RAG")

test_queries = [
    "What is LCEL and how do I use it?",
    "Explain different types of memory in conversational AI",
    "How do retrievers work in RAG applications?",
]

for i, query in enumerate(test_queries, 1):
    print("\n" + "=" * 80)
    print(f"Query {i}: {query}")
    print("=" * 80)
    
    start_time = time.time()
    response = fusion_rag_chain.invoke(query)
    elapsed = time.time() - start_time
    
    print("\n" + response)
    print(f"\n⏱️  Time: {elapsed:.2f}s")


TESTING FUSION RAG


Query 1: What is LCEL and how do I use it?

Based on the provided documents, there is no information about **LCEL (LangChain Expression Language)**. The retrieved content focuses on LangChain’s core benefits, agents, RAG chains, and model interfaces, but does not mention LCEL or its usage.

Please provide additional context or documents that include LCEL, and I’ll be happy to answer based on them.

⏱️  Time: 6.67s

Query 2: Explain different types of memory in conversational AI

Based on the retrieved documents, conversational AI systems utilize at least three types of memory, each serving a distinct purpose:

1. **Short-term memory**  
   Handles the immediate context of a single turn or the most recent exchanges. It is typically implemented using a sliding window of recent messages or token limits. This allows the model to remember what was just said without storing full history.

2. **Conversational (multi-turn) memory**  
   Used to support **multi-turn intera

## 8. 详细示例：检查融合过程

In [8]:
print_section_header("Detailed Fusion Process Analysis")

query = "What are the main components of LangChain?"
print(f"Query: {query}\n")

# Run with verbose mode
print("=" * 80)
print("FUSION RETRIEVAL PROCESS:")
print("=" * 80)

fused_docs = fusion_retriever(
    query=query,
    retriever=retriever,
    llm=llm,
    num_queries=4,
    k_final=6,
    verbose=True,
)

print("\n" + "=" * 80)
print("FINAL RETRIEVED DOCUMENTS:")
print("=" * 80)
for i, doc in enumerate(fused_docs, 1):
    print(f"\nDocument {i}:")
    print(f"Source: {doc.metadata.get('source', 'unknown')[:60]}")
    print(f"Content: {doc.page_content[:200]}...")


DETAILED FUSION PROCESS ANALYSIS

Query: What are the main components of LangChain?

FUSION RETRIEVAL PROCESS:

[Fusion Retriever] Original query: What are the main components of LangChain?

[Fusion Retriever] Generated 4 queries:
  1. What are the main components of LangChain?
  2. What are the core building blocks or modules that constitute the LangChain framework?
  3. Can you detail the main subcomponents of LangChain, including models, prompts, chains, memory, agents, and tools?
  4. In the context of building applications with large language models, what are the key structural elements of the LangChain library?

[Fusion Retriever] Query 'What are the main components of LangChain?...' retrieved 4 docs

[Fusion Retriever] Query 'What are the core building blocks or modules that ...' retrieved 4 docs

[Fusion Retriever] Query 'Can you detail the main subcomponents of LangChain...' retrieved 4 docs

[Fusion Retriever] Query 'In the context of building applications with large...' ret

## 9. 对比：标准 vs 多查询 vs Fusion RAG

In [9]:
# Build comparison chains
from shared.prompts import RAG_PROMPT_TEMPLATE, MULTI_QUERY_PROMPT

print_section_header("Comparison: Standard vs Multi-Query vs Fusion RAG")

# Standard RAG
standard_chain = (
    {"context": retriever | format_docs, "input": RunnablePassthrough()}
    | RAG_PROMPT_TEMPLATE
    | llm
    | StrOutputParser()
)

# Multi-Query RAG (simple deduplication)
def multi_query_retriever_simple(query: str) -> List[Document]:
    """Multi-Query RAG with simple deduplication."""
    # Generate alternatives
    multi_query_chain = MULTI_QUERY_PROMPT | llm | StrOutputParser()
    alternatives = multi_query_chain.invoke({"question": query})
    alternative_queries = [
        line.strip() for line in alternatives.split("\n") if line.strip()
    ]
    
    # Retrieve for each query
    all_docs = []
    seen_content = set()
    
    for q in [query] + alternative_queries[:2]:
        docs = retriever.invoke(q)
        for doc in docs:
            if doc.page_content not in seen_content:
                all_docs.append(doc)
                seen_content.add(doc.page_content)
    
    return all_docs[:6]

multi_query_chain = (
    {
        "context": RunnableLambda(multi_query_retriever_simple) | format_docs,
        "input": RunnablePassthrough(),
    }
    | RAG_PROMPT_TEMPLATE
    | llm
    | StrOutputParser()
)

# Test query
test_query = "How do I build chains in LangChain?"

print(f"\nQuery: {test_query}\n")
print("=" * 80)

# Standard RAG
print("\n[1] STANDARD RAG")
print("-" * 80)
start = time.time()
response_standard = standard_chain.invoke(test_query)
time_standard = time.time() - start
print(response_standard)
print(f"\n⏱️  Time: {time_standard:.2f}s")

# Multi-Query RAG
print("\n" + "=" * 80)
print("\n[2] MULTI-QUERY RAG (Simple Deduplication)")
print("-" * 80)
start = time.time()
response_multi = multi_query_chain.invoke(test_query)
time_multi = time.time() - start
print(response_multi)
print(f"\n⏱️  Time: {time_multi:.2f}s")

# Fusion RAG
print("\n" + "=" * 80)
print("\n[3] FUSION RAG (Reciprocal Rank Fusion)")
print("-" * 80)
start = time.time()
response_fusion = fusion_rag_chain.invoke(test_query)
time_fusion = time.time() - start
print(response_fusion)
print(f"\n⏱️  Time: {time_fusion:.2f}s")

# Summary
print("\n" + "=" * 80)
print("PERFORMANCE SUMMARY:")
print("=" * 80)
print(f"Standard RAG:     {time_standard:.2f}s (baseline)")
print(f"Multi-Query RAG:  {time_multi:.2f}s ({time_multi/time_standard:.1f}x slower)")
print(f"Fusion RAG:       {time_fusion:.2f}s ({time_fusion/time_standard:.1f}x slower)")


COMPARISON: STANDARD VS MULTI-QUERY VS FUSION RAG


Query: How do I build chains in LangChain?


[1] STANDARD RAG
--------------------------------------------------------------------------------
Based on the context provided, there is not enough information to answer your question about how to build chains in LangChain. The context mentions "chains" only in a list of core components under the LangChain overview, but does not include any details, instructions, examples, or code snippets for building chains.

The provided context mainly covers:
- Updating model profiles through pull requests and CLI tools
- Setting up LangSmith for tracing and observability
- A high-level overview of LangChain (mentioning agents, models, tools, etc.)

To learn how to build chains, I recommend referring to the LangChain documentation's "Chains" section or the "Get started" guide, which are not included in the context above.

⏱️  Time: 4.33s


[2] MULTI-QUERY RAG (Simple Deduplication)
-------------------

## 10. RRF 参数调优

探索不同的 RRF 参数如何影响排名。

In [10]:
print_section_header("RRF Parameter Tuning")

query = "What is LCEL?"
print(f"Query: {query}\n")

# Test different k values
k_values = [10, 60, 100]

print("Testing different RRF k values:")
print("=" * 80)

for k in k_values:
    print(f"\nk = {k}:")
    print("-" * 80)
    
    docs = fusion_retriever(
        query=query,
        retriever=retriever,
        llm=llm,
        num_queries=3,
        k_final=3,
        rrf_k=k,
        verbose=False,
    )

    for i, doc in enumerate(docs, 1):
        print(f"  {i}. {doc.page_content[:80]}...")

print("\n" + "=" * 80)
print("OBSERVATIONS:")
print("=" * 80)
print("• Lower k (e.g., 10): More weight on top-ranked documents")
print("• Higher k (e.g., 100): More uniform weighting across ranks")
print("• Standard k = 60: Good balance (recommended in literature)")
print("• Results often similar due to robust algorithm")


RRF PARAMETER TUNING

Query: What is LCEL?

Testing different RRF k values:

k = 10:
--------------------------------------------------------------------------------
  1. documents2. Retrieval and generationRAG agentsRAG chainsSecurity: indirect promp...
  2. LangChain overview - Docs by LangChainSkip to main contentJoin us May 13th & May...
  3. ​ Core benefits
Standard model interfaceDifferent providers have unique APIs for...

k = 60:
--------------------------------------------------------------------------------
  1. Build a RAG agent with LangChain - Docs by LangChainSkip to main contentJoin us ...
  2. LangChain overview - Docs by LangChainSkip to main contentJoin us May 13th & May...
  3. documents2. Retrieval and generationRAG agentsRAG chainsSecurity: indirect promp...

k = 100:
--------------------------------------------------------------------------------
  1. documents2. Retrieval and generationRAG agentsRAG chainsSecurity: indirect promp...
  2. LangChain overview - Doc

## 11. 性能指标

In [11]:
print_section_header("Performance Metrics")

num_queries_generated = 3  # Alternative queries (+ 1 original = 4 total)
retrievals_per_query = 4
total_retrievals = 4  # num_queries

print("\nCOST BREAKDOWN:")
print("-" * 80)
print(f"Query generation: {num_queries_generated} LLM calls")
print(f"Retrievals: {total_retrievals} vector searches")
print(f"Documents retrieved (total): ~{total_retrievals * retrievals_per_query}")
print(f"Documents after RRF: 6 (deduplicated + reranked)")
print(f"Final generation: 1 LLM call")
print(f"\nTotal LLM calls: {num_queries_generated + 1}")
print(f"Total vector searches: {total_retrievals}")

print("\n" + "=" * 80)
print("COMPARISON WITH OTHER APPROACHES:")
print("=" * 80)
print("\nLLM Calls:")
print("  • Standard RAG: 1 (generation only)")
print("  • Multi-Query RAG: 4 (1 gen + 3 query gen)")
print("  • Fusion RAG: 4 (1 gen + 3 query gen)")
print("\nVector Searches:")
print("  • Standard RAG: 1")
print("  • Multi-Query RAG: 3-4")
print("  • Fusion RAG: 4")
print("\nRanking Quality:")
print("  • Standard RAG: ⭐⭐⭐ (baseline)")
print("  • Multi-Query RAG: ⭐⭐⭐⭐ (better coverage)")
print("  • Fusion RAG: ⭐⭐⭐⭐⭐ (best ranking)")


PERFORMANCE METRICS


COST BREAKDOWN:
--------------------------------------------------------------------------------
Query generation: 3 LLM calls
Retrievals: 4 vector searches
Documents retrieved (total): ~16
Documents after RRF: 6 (deduplicated + reranked)
Final generation: 1 LLM call

Total LLM calls: 4
Total vector searches: 4

COMPARISON WITH OTHER APPROACHES:

LLM Calls:
  • Standard RAG: 1 (generation only)
  • Multi-Query RAG: 4 (1 gen + 3 query gen)
  • Fusion RAG: 4 (1 gen + 3 query gen)

Vector Searches:
  • Standard RAG: 1
  • Multi-Query RAG: 3-4
  • Fusion RAG: 4

Ranking Quality:
  • Standard RAG: ⭐⭐⭐ (baseline)
  • Multi-Query RAG: ⭐⭐⭐⭐ (better coverage)
  • Fusion RAG: ⭐⭐⭐⭐⭐ (best ranking)


## 12. 关键要点

### 总结

**Fusion RAG** 通过使用倒数排名融合改进了多查询 RAG：
- 生成多个查询变体（类似多查询）
- 应用复杂的 RRF 算法进行排名
- 保留并合并排名信息
- 比简单去重更稳健

### RRF 算法优势

✅ **为什么 RRF 有效：**
- 有利于出现在多个结果中的文档
- 平衡频率和排名位置
- 无需分数归一化
- 对异常值和变化具有鲁棒性
- 在信息检索中被证明有效

### 成本效益分析

| 方面 | 影响 | 说明 |
|--------|--------|-------|
| **查询延迟** | ❌ 慢 2-3 倍 | 多次检索 |
| **LLM 调用** | ❌ +3-4 次调用 | 查询生成 |
| **检索质量** | ✅ 最佳 | RRF 排名 |
| **覆盖率** | ✅ 优秀 | 多视角 |
| **鲁棒性** | ✅ 高 | 对措辞不太敏感 |
| **实现** | ⚠️ 中等 | 更复杂 |

### 最佳实践

1. **查询数量**：3-5 个查询是最佳的（更多 = 收益递减）
2. **RRF 常数 (k)**：60 是标准值，根据你的用例进行调整
3. **最终 k**：返回比所需更多的文档，让 RRF 决定质量
4. **缓存**：为常见查询缓存查询生成
5. **监控**：跟踪哪些查询从融合中受益最多

### 何时使用

选择 **Fusion RAG** 当：
- ✅ 查询质量比速度更重要
- ✅ 处理复杂的、多层面的问题
- ✅ 需要在查询变体中进行稳健的检索
- ✅ 可以承受额外的延迟和成本

选择 **多查询 RAG** 当：
- ✅ 偏好更简单的实现
- ✅ 去重已足够
- ✅ 希望稍微降低成本

选择 **标准 RAG** 当：
- ✅ 速度至关重要
- ✅ 简单查询
- ✅ 预算限制紧张

### 扩展

- **混合融合**：与关键词搜索结合
- **加权查询**：为不同类型的查询分配不同权重
- **查询过滤**：移除低质量的生成查询
- **自适应融合**：仅对复杂查询使用融合

---

**复杂度评级:** ⭐⭐⭐ (中等 - RRF 算法增加了复杂性)

**生产就绪度:** ⭐⭐⭐⭐ (高 - 经过验证的算法，可管理的权衡)

继续到 **14_sql_rag.ipynb** 了解自然语言转 SQL RAG！